# 02_load_traffic_counts
Loads and cleans 2025 NYC traffic counts. Saves `traffic_counts_clean`.

In [ ]:
import requests
import pandas as pd
from pyspark.sql import functions as F

SCHEMA = "workspace.default"
YEAR = 2025
URL = "https://data.cityofnewyork.us/resource/7ym2-wayt.json"  # traffic counts

In [ ]:
# fetch API
rows, page = [], 50000
while True:
    r = requests.get(URL, params={"$where": f"yr={YEAR}", "$order": ":id",
                                  "$limit": page, "$offset": len(rows)}, timeout=120)
    r.raise_for_status()
    batch = r.json()
    rows += batch
    if len(batch) < page:
        break
pdf = pd.DataFrame(rows).astype(object)
pdf = pdf.where(pdf.notna(), None)
raw = spark.createDataFrame(pdf, schema=", ".join(f"{c} string" for c in pdf.columns))
print(len(rows))
raw.printSchema()

In [ ]:
# clean
point = r"POINT \(([-0-9.]+) ([-0-9.]+)\)"
n0 = raw.count()
t = raw.select(
    F.col("requestid").cast("int").alias("request_id"),
    "boro",
    F.col("segmentid").cast("int").alias("segment_id"),
    "street",
    F.col("fromst").alias("from_st"),
    F.col("tost").alias("to_st"),
    "direction",
    F.col("vol").cast("int").alias("vol"),
    F.make_timestamp(*[F.col(c).cast("int") for c in ["yr", "m", "d", "hh", "mm"]], F.lit(0)).alias("ts"),
    *[F.when(F.regexp_extract("wktgeom", point, i) != "", F.regexp_extract("wktgeom", point, i))
       .cast("double").alias(name) for i, name in [(1, "x_ft"), (2, "y_ft")]])
t = t.dropna(subset=["segment_id", "vol", "ts", "x_ft", "y_ft"]).filter(F.col("vol") >= 0)
n1 = t.count()
t = t.dropDuplicates(["segment_id", "direction", "ts"])
n2 = t.count()
t = (t.withColumn("date", F.to_date("ts"))
      .withColumn("hour", F.hour("ts"))
      .withColumn("dow", F.dayofweek("ts")))
print(n0, n1, n2)
t.write.mode("overwrite").saveAsTable(f"{SCHEMA}.traffic_counts_clean")

In [ ]:
spark.table(f"{SCHEMA}.traffic_counts_clean").count()